# Exploration : Remplissage des matchs Paris Basketball

Question métier : quels matchs sous-performent en remplissage, et quels
leviers (tarification, campagnes, timing) pourraient corriger ça ?

Ce notebook explore `gold.fact_match`, construite par le pipeline
(`pipeline/gold/build_gold.py`), qui contient une ligne par match à domicile
avec le taux de remplissage, le contexte sportif, temporel et météo.

In [ ]:
import os
os.environ.setdefault("MPLBACKEND", "Agg")

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

con = duckdb.connect("../data/warehouse/pbb.duckdb")
df = con.execute("SELECT * FROM gold.fact_match ORDER BY date").fetchdf()
print(f"{len(df)} matchs chargés")
df.head()

## 1. Vue d'ensemble du remplissage

In [ ]:
print(df['taux_remplissage'].describe())
print()
print("Par compétition :")
print(df.groupby('competition')['taux_remplissage'].agg(['mean', 'min', 'max', 'count']).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='competition', y='taux_remplissage', ax=ax)
ax.set_title("Taux de remplissage par compétition")
ax.set_ylabel("Taux de remplissage")
ax.set_xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("../dashboard/fig_remplissage_par_competition.png", dpi=120)
plt.show()

## 2. Les matchs les plus faibles

Les 8 matchs avec le plus faible taux de remplissage, et leur contexte.

In [ ]:
cols = ['match_id', 'date', 'adversaire', 'competition', 'taux_remplissage',
        'rang_adversaire_avant', 'jour_semaine', 'vacances_scolaires', 'temperature']
df.nsmallest(8, 'taux_remplissage')[cols]

## 3. Le classement de l'adversaire a-t-il un effet ?

Hypothèse formée à l'oeil sur les données : un adversaire mal classé
(rang élevé) semble associé à un taux de remplissage plus faible.

In [ ]:
corr = df[['taux_remplissage', 'rang_adversaire_avant', 'rang_pbb_avant', 'ecart_classement']].corr()
print(corr['taux_remplissage'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=df, x='rang_adversaire_avant', y='taux_remplissage', ax=ax,
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
ax.set_title("Taux de remplissage vs classement de l'adversaire avant match")
ax.set_xlabel("Rang de l'adversaire avant le match (1 = meilleur)")
ax.set_ylabel("Taux de remplissage")
plt.tight_layout()
plt.savefig("../dashboard/fig_remplissage_vs_classement.png", dpi=120)
plt.show()

## 4. Jour de semaine et vacances scolaires

In [ ]:
print(df.groupby('jour_semaine')['taux_remplissage'].agg(['mean', 'count']).round(3).sort_values('mean'))
print()
print(df.groupby('vacances_scolaires')['taux_remplissage'].agg(['mean', 'count']).round(3))

## 5. Prix moyen du billet vs remplissage

Le prix a-t-il été ajusté à la baisse pour les matchs à faible affluence
anticipée, ou est-ce un effet a posteriori (moins de monde = moins
d'abonnés, qui paient un tarif différent) ?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x='prix_moyen_billet', y='taux_remplissage', hue='competition', ax=ax)
ax.set_title("Prix moyen du billet vs taux de remplissage")
plt.tight_layout()
plt.savefig("../dashboard/fig_prix_vs_remplissage.png", dpi=120)
plt.show()

## 6. Synthèse

*(à compléter avec les vraies conclusions une fois exécuté sur les données
complètes -- voir NOTES.md pour la version finale destinée à la
soutenance)*

Pistes à date sur l'échantillon exploré :
- Le classement de l'adversaire semble être le facteur le plus discriminant
- L'EuroLeague remplit mieux que le championnat national en moyenne
- À vérifier avec les données complètes : effet jour de semaine, vacances,
  météo -- les patterns observés sur un sous-ensemble peuvent ne pas se
  confirmer sur la saison complète.